# SETI GPU Backend

Upload this notebook to Google Colab, select **Runtime > Change runtime type > T4 GPU**, and run every cell.

- Uses no GitHub account or credentials.
- Accepts only public `https://bldata.berkeley.edu` HDF5 URLs.
- Rejects files larger than 1 GB.
- Deletes the temporary HDF5 after each analysis.
- Generates a random one-session Bearer token.
- Statistical candidates require independent verification.

In [ ]:
!pip install -q fastapi uvicorn h5py numpy requests torch

import os
import re
import secrets
import tempfile
import threading
import subprocess
import time
from datetime import datetime, timezone
from urllib.parse import urlparse

import h5py
import numpy as np
import requests
import torch
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware

MAX_DOWNLOAD_BYTES = 1024 * 1024 * 1024
MAX_BINS = 4096
SESSION_TOKEN = secrets.token_urlsafe(24)

print("CUDA GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
app = FastAPI(title="SETI GPU Backend", docs_url=None, redoc_url=None)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["GET", "POST", "OPTIONS"],
    allow_headers=["Authorization", "Content-Type"],
)

def require_token(authorization):
    if authorization != "Bearer " + SESSION_TOKEN:
        raise HTTPException(status_code=401, detail="Invalid or missing Bearer token")

def validate_url(url):
    parsed = urlparse(url)
    if parsed.scheme != "https" or parsed.hostname != "bldata.berkeley.edu":
        raise ValueError("Only HTTPS public files at bldata.berkeley.edu are accepted")

def download_h5(url):
    validate_url(url)
    response = requests.get(url, stream=True, timeout=60, allow_redirects=True)
    response.raise_for_status()
    length = int(response.headers.get("Content-Length", "0"))
    if length <= 0 or length > MAX_DOWNLOAD_BYTES:
        response.close()
        raise ValueError("HDF5 file must have a known size and must not exceed 1 GB")

    fd, path = tempfile.mkstemp(suffix=".h5")
    downloaded = 0
    try:
        with os.fdopen(fd, "wb") as output:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if not chunk:
                    continue
                downloaded += len(chunk)
                if downloaded > MAX_DOWNLOAD_BYTES:
                    raise ValueError("HDF5 download exceeded 1 GB")
                output.write(chunk)
    except Exception:
        response.close()
        os.unlink(path)
        raise
    response.close()
    return path, downloaded

@app.get("/api/health")
async def health(authorization: str = Header(None)):
    require_token(authorization)
    return {"status": "online", "gpu": torch.cuda.is_available(), "backend": "CUDA/PyTorch" if torch.cuda.is_available() else "Unavailable"}

@app.post("/api/gpu_hits")
async def gpu_hits(payload: dict, authorization: str = Header(None)):
    require_token(authorization)
    if not torch.cuda.is_available():
        raise HTTPException(status_code=400, detail="CUDA GPU is unavailable. Enable a GPU runtime and rerun.")

    url = payload.get("url")
    if not isinstance(url, str):
        raise HTTPException(status_code=400, detail="url is required")
    start_bin = int(payload.get("start_bin", 0))
    num_bins = int(payload.get("num_bins", 256))
    threshold = float(payload.get("threshold", 6.5))
    if start_bin < 0 or not 1 <= num_bins <= MAX_BINS:
        raise HTTPException(status_code=400, detail="Invalid bin selection")

    path = None
    try:
        path, downloaded = download_h5(url)
        with h5py.File(path, "r") as h5:
            dataset = h5["data"]
            shape = list(dataset.shape)

            if len(shape) == 2:
                if shape[1] >= num_bins:
                    end = min(start_bin + num_bins, shape[1])
                    matrix = np.asarray(dataset[:, start_bin:end], dtype=np.float32)
                else:
                    end = min(start_bin + num_bins, shape[0])
                    matrix = np.asarray(dataset[start_bin:end, :], dtype=np.float32)
            elif len(shape) == 3:
                labels = []
                if "DIMENSION_LABELS" in dataset.attrs:
                    labels = [str(v).lower() for v in dataset.attrs["DIMENSION_LABELS"]]
                if "time" in labels and "frequency" in labels:
                    frequency_axis = labels.index("frequency")
                    time_axis = labels.index("time")
                    end = min(start_bin + num_bins, shape[frequency_axis])
                    selectors = [slice(None)] * 3
                    selectors[frequency_axis] = slice(start_bin, end)
                    selectors[time_axis] = slice(0, min(277, shape[time_axis]))
                    matrix = np.asarray(dataset[tuple(selectors)], dtype=np.float32)
                    matrix = np.moveaxis(matrix, frequency_axis, -1).reshape(-1, end - start_bin)
                else:
                    end = min(start_bin + num_bins, shape[2])
                    matrix = np.asarray(dataset[:277, 0, start_bin:end], dtype=np.float32)
            elif len(shape) == 4:
                end = min(start_bin + num_bins, shape[3])
                matrix = np.asarray(dataset[0, :277, 0, start_bin:end], dtype=np.float32)
            else:
                raise ValueError("Unsupported HDF5 data shape: " + str(shape))

            fch1 = float(dataset.attrs.get("fch1", 1420.405))
            foff = float(dataset.attrs.get("foff", -0.00286102294921875))

        values = torch.as_tensor(matrix, device="cuda", dtype=torch.float32)
        median = torch.median(values, dim=0).values
        mad = torch.median(torch.abs(values - median), dim=0).values
        noise = torch.clamp(mad * 1.4826, min=1e-6)
        snr = (values.max(dim=0).values - median) / noise
        peak_time = values.argmax(dim=0).values
        snr = snr.detach().cpu().numpy()
        peak_time = peak_time.detach().cpu().numpy()

        peaks = []
        for channel in np.argsort(snr)[::-1][:20]:
            score = float(snr[channel])
            if score >= threshold:
                peaks.append({"channel": int(channel), "snr": round(score, 3), "time_bin": int(peak_time[channel])})

        spectrum = np.mean(matrix, axis=0)
        spread = max(float(np.max(spectrum) - np.min(spectrum)), 1e-6)
        spectrum = (spectrum - np.min(spectrum)) / spread
        timestamp = datetime.now(timezone.utc).isoformat()
        candidates = []
        for peak in peaks[:50]:
            candidates.append({
                "time": timestamp,
                "frequency_mhz": round(fch1 + (start_bin + peak["channel"]) * foff, 6),
                "freq_shift": f"{peak['channel'] * foff * 1e6:+.2f} Hz/ch",
                "drift_rate": "0.00 Hz/s",
                "classification": "Narrowband candidate",
                "snr": peak["snr"],
                "time_bin": peak["time_bin"],
                "status": "Candidate - requires verification"
            })

        return {
            "status": "online",
            "source": url,
            "source_shape": shape,
            "downloaded_bytes": downloaded,
            "max_download_bytes": MAX_DOWNLOAD_BYTES,
            "matrix_shape": list(matrix.shape),
            "start_bin": start_bin,
            "num_bins": num_bins,
            "center_frequency_mhz": fch1,
            "channel_spacing_mhz": foff,
            "snr_peak": peaks[0]["snr"] if peaks else 0.0,
            "snr_threshold": threshold,
            "accelerated": True,
            "backend": "CUDA/PyTorch",
            "spectrum_row": np.round(spectrum * 255).astype(int).tolist(),
            "candidates": candidates,
            "iso_timestamp": timestamp
        }
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc))
    except requests.RequestException as exc:
        raise HTTPException(status_code=502, detail="Public HDF5 source request failed")
    finally:
        if path and os.path.exists(path):
            os.unlink(path)

## Start the temporary secure tunnel

Keep this runtime open. Copy the API URL and Bearer token into the SETI website. Do not commit or publicly post the token.

In [ ]:
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

server = threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"), daemon=True)
server.start()

tunnel = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
deadline = time.time() + 30
while time.time() < deadline and public_url is None:
    line = tunnel.stdout.readline()
    match = re.search(r"https://[^ ]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)

print("\n=== SETI GPU CONNECTION ===")
print("API URL:", public_url or "Tunnel still starting - rerun this cell if needed")
print("Bearer token:", SESSION_TOKEN)
print("Treat the token like a password. It expires when this runtime ends.")
print("===========================\n")